In [ ]:
import bw2io as bi # ensemble des fonctions et classes pour importer et exporter (input/output)
import bw2data as bd # ... pour gérer les données du projet
import bw2calc as bc # ... pour faire des opérations
import bw2analyzer as ba # ... pour interpréter les résultats

import pandas as pd # pour utiliser un format de table pratique
import seaborn as sns # pour tracer des graphes à partir de ces tables
import matplotlib.pyplot as plt

from pathlib import Path

In [ ]:
# Nom de projet et bases de données requises
PROJECT = "project_ecoinvent_311"
REQUIRED_DATABASES = {"ecoinvent-3.11-biosphere", "ecoinvent-3.11-cutoff"}

# Chemin vers archive contenant ecoinvent 3.11 cutoff
BACKUP_LOCATIONS = [
    # JupyterHub
    Path("/srv/cours-acv-2026/brightway2_project_ecoinvent_311.tar.gz"),

    # Local computer
    Path.cwd().joinpath("brightway2_project_ecoinvent_311.tar.gz")
]
BACKUP = next((path for path in BACKUP_LOCATIONS if path.exists()), None)

# Vérification de l'état du projet
project_is_ready = False
if PROJECT in bd.projects:
    bd.projects.set_current(PROJECT)
    missing_databases = REQUIRED_DATABASES - set(bd.databases)
    if not missing_databases:
        project_is_ready = True
        print(f"Le projet Brightway '{PROJECT}' est prêt.")
    else:
        print(f"Le projet '{PROJECT}' existe mais est incomplet.")
        print("Base(s) de donnée(s) manquante(s):")
        for db in sorted(missing_databases):
            print(f"  - {db}")

# Restore projet si besoin
if not project_is_ready:
    if PROJECT in bd.projects:
        print(f"Suppression du projet incomplet '{PROJECT}'...")

        bd.projects.delete_project(
            PROJECT,
            delete_dir=True
        )
        bd.projects.purge_deleted_directories()

    print(f"Installation du projet depuis:\n{BACKUP}")

    bi.backup.restore_project_directory(
        fp=BACKUP,
        project_name=PROJECT,
        switch=True,
    )

    missing_databases = REQUIRED_DATABASES - set(bd.databases)

    if missing_databases:
        raise RuntimeError(
            "Projet restoré, mais des bases de données "
            f"manquent toujours: {sorted(missing_databases)}"
        )

    print(f"Le projet '{PROJECT}' a été installé avec succès.")


# Activation du projet
bd.projects.set_current(PROJECT)
print(f"\nProjet activé: {bd.projects.current}")